# Remove facial keypoints and add yoga joint angles

This notebook reads the cleaned normalized train/test keypoint files, keeps nose keypoint `0`, removes facial keypoints `1` through `10`, and writes two new CSV files directly under `csv_data`.

Angles are calculated in 3D and reported in degrees. When an angle cannot be calculated because a required landmark is missing, it is filled with that pose's mean angle learned from the training set. The training-wide angle mean is used only as a fallback if a pose-specific mean is unavailable.

## Angle definitions

- Left/right shoulder: elbow–shoulder–hip
- Left/right hip: shoulder–hip–knee
- Left/right knee: hip–knee–ankle
- Knee-to-hip-center-to-knee: left knee–midpoint of the two hips–right knee

MediaPipe landmark indices: shoulders `11/12`, elbows `13/14`, hips `23/24`, knees `25/26`, and ankles `27/28`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start: Path = Path.cwd()) -> Path:
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / 'csv_data' / 'normalized_noempty').is_dir():
            return directory
    raise FileNotFoundError(
        'Could not find csv_data/normalized_noempty from the current directory.'
    )


PROJECT_ROOT = find_project_root()
INPUT_DIR = PROJECT_ROOT / 'csv_data' / 'normalized_noempty'
OUTPUT_DIR = PROJECT_ROOT / 'csv_data'

INPUT_FILES = {
    'train': INPUT_DIR / 'keypoints_train.csv',
    'test': INPUT_DIR / 'keypoints_test.csv',
}
OUTPUT_FILES = {
    'train': OUTPUT_DIR / 'keypoints_train_body_angles.csv',
    'test': OUTPUT_DIR / 'keypoints_test_body_angles.csv',
}

PROJECT_ROOT

WindowsPath('C:/Users/vgohu/Desktop/yoga_project')

In [2]:
COORDINATES = ('x', 'y', 'z')
FACIAL_KEYPOINTS_TO_REMOVE = range(1, 11)
ANGLE_DEFINITIONS = {
    'left_shoulder_angle_deg': (13, 11, 23),
    'right_shoulder_angle_deg': (14, 12, 24),
    'left_hip_angle_deg': (11, 23, 25),
    'right_hip_angle_deg': (12, 24, 26),
    'left_knee_angle_deg': (23, 25, 27),
    'right_knee_angle_deg': (24, 26, 28),
}
ANGLE_COLUMNS = [*ANGLE_DEFINITIONS, 'knee_hip_center_knee_angle_deg']


def landmark_array(dataframe: pd.DataFrame, keypoint: int) -> np.ndarray:
    columns = [f'kp_{keypoint}_{coordinate}' for coordinate in COORDINATES]
    return dataframe[columns].to_numpy(dtype=float)


def valid_landmarks(points: np.ndarray) -> np.ndarray:
    return np.isfinite(points).all(axis=1) & ~np.isclose(points, 0.0).all(axis=1)


def angle_degrees(
    point_a: np.ndarray,
    vertex: np.ndarray,
    point_c: np.ndarray,
    *,
    allow_zero_vertex: bool = False,
) -> np.ndarray:
    vector_a = point_a - vertex
    vector_c = point_c - vertex
    norm_product = np.linalg.norm(vector_a, axis=1) * np.linalg.norm(vector_c, axis=1)

    valid_vertex = (
        np.isfinite(vertex).all(axis=1)
        if allow_zero_vertex
        else valid_landmarks(vertex)
    )
    valid = (
        valid_landmarks(point_a)
        & valid_vertex
        & valid_landmarks(point_c)
        & np.isfinite(norm_product)
        & (norm_product > 0.0)
    )
    angles = np.full(len(point_a), np.nan, dtype=float)
    cosine = np.einsum('ij,ij->i', vector_a[valid], vector_c[valid]) / norm_product[valid]
    angles[valid] = np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))
    return angles


def transform_keypoints(dataframe: pd.DataFrame) -> pd.DataFrame:
    result = dataframe.copy()

    required_columns = {
        f'kp_{keypoint}_{coordinate}'
        for keypoint in [0, 11, 12, 13, 14, 23, 24, 25, 26, 27, 28]
        for coordinate in COORDINATES
    }
    missing_columns = sorted(required_columns.difference(result.columns))
    if missing_columns:
        raise ValueError(f'Missing required columns: {missing_columns}')

    for angle_name, (point_a, vertex, point_c) in ANGLE_DEFINITIONS.items():
        result[angle_name] = angle_degrees(
            landmark_array(result, point_a),
            landmark_array(result, vertex),
            landmark_array(result, point_c),
        )

    left_hip = landmark_array(result, 23)
    right_hip = landmark_array(result, 24)
    hip_midpoint = (left_hip + right_hip) / 2.0
    valid_hips = valid_landmarks(left_hip) & valid_landmarks(right_hip)
    hip_midpoint[~valid_hips] = np.nan
    result['knee_hip_center_knee_angle_deg'] = angle_degrees(
        landmark_array(result, 25),
        hip_midpoint,
        landmark_array(result, 26),
        allow_zero_vertex=True,
    )

    facial_columns = [
        f'kp_{keypoint}_{coordinate}'
        for keypoint in FACIAL_KEYPOINTS_TO_REMOVE
        for coordinate in COORDINATES
    ]
    result = result.drop(columns=facial_columns)

    metadata_columns = [column for column in ('center_type', 'label') if column in result.columns]
    feature_columns = [
        column for column in result.columns
        if column not in ANGLE_COLUMNS and column not in metadata_columns
    ]
    return result[feature_columns + ANGLE_COLUMNS + metadata_columns]


def fill_missing_angles_from_training_pose_means(
    dataframe: pd.DataFrame,
    pose_means: pd.DataFrame,
    global_means: pd.Series,
) -> pd.DataFrame:
    result = dataframe.copy()
    for angle_column in ANGLE_COLUMNS:
        mean_for_each_row = result['label'].map(pose_means[angle_column])
        result[angle_column] = (
            result[angle_column]
            .fillna(mean_for_each_row)
            .fillna(global_means[angle_column])
        )
    return result


In [3]:
source_datasets = {
    split: pd.read_csv(input_path)
    for split, input_path in INPUT_FILES.items()
}
raw_transformed_datasets = {
    split: transform_keypoints(dataframe)
    for split, dataframe in source_datasets.items()
}

# Fit imputation values on training data only to avoid test-set leakage.
train_pose_angle_means = (
    raw_transformed_datasets['train']
    .groupby('label', sort=True)[ANGLE_COLUMNS]
    .mean()
)
train_global_angle_means = raw_transformed_datasets['train'][ANGLE_COLUMNS].mean()

reports = []
transformed_datasets = {}
for split, raw_transformed in raw_transformed_datasets.items():
    missing_before = int(raw_transformed[ANGLE_COLUMNS].isna().sum().sum())
    transformed = fill_missing_angles_from_training_pose_means(
        raw_transformed, train_pose_angle_means, train_global_angle_means
    )
    transformed.to_csv(OUTPUT_FILES[split], index=False)
    transformed_datasets[split] = transformed

    reports.append({
        'split': split,
        'rows': len(transformed),
        'source_columns': len(source_datasets[split].columns),
        'output_columns': len(transformed.columns),
        'angles_imputed': missing_before,
        'output_path': str(OUTPUT_FILES[split]),
    })

pd.DataFrame(reports)

,split,rows,source_columns,output_columns,angles_imputed,output_path
0,train,1042,104,81,701,C:\Users\vgohu\Desktop\yoga_project\csv_data\k...
1,test,465,104,81,381,C:\Users\vgohu\Desktop\yoga_project\csv_data\k...


In [4]:
removed_columns = {
    f'kp_{keypoint}_{coordinate}'
    for keypoint in FACIAL_KEYPOINTS_TO_REMOVE
    for coordinate in COORDINATES
}

for split, dataframe in transformed_datasets.items():
    assert {'kp_0_x', 'kp_0_y', 'kp_0_z'}.issubset(dataframe.columns)
    assert removed_columns.isdisjoint(dataframe.columns)
    assert set(ANGLE_COLUMNS).issubset(dataframe.columns)
    assert not dataframe[ANGLE_COLUMNS].isna().any().any()
    angles = dataframe[ANGLE_COLUMNS].to_numpy(dtype=float)
    assert np.isfinite(angles).all()
    assert ((angles >= 0.0) & (angles <= 180.0)).all()
    assert OUTPUT_FILES[split].is_file()

pd.concat(
    {split: dataframe[ANGLE_COLUMNS].notna().sum() for split, dataframe in transformed_datasets.items()},
    axis=1,
).rename_axis('angle').rename(columns=str.title)

,Train,Test
angle,,
left_shoulder_angle_deg,1042,465
right_shoulder_angle_deg,1042,465
left_hip_angle_deg,1042,465
right_hip_angle_deg,1042,465
left_knee_angle_deg,1042,465
right_knee_angle_deg,1042,465
knee_hip_center_knee_angle_deg,1042,465


## Outputs

- `csv_data/keypoints_train_body_angles.csv`
- `csv_data/keypoints_test_body_angles.csv`